# 05 - Data Quality
Validações automatizadas (schema, null ratio, unicidade, intervalos, valores aceitos) e relatório JSON em `reports/data_quality_report.json`.
**ERRO** reprova o pipeline. **AVISO** só registra.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import json
import logging
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("src").setLevel(logging.INFO)

from src.config import load_settings
from src.quality import checks
from src.quality.report import build_report, count_tables, raise_if_failed, save_report
from src.quality.rules import run_all_checks

os.environ["FOOTBALL_DATA_API_KEY"] = dbutils.secrets.get(scope="fc", key="football_data_api_key")
settings = load_settings()

In [0]:
resultados = run_all_checks(spark, settings)
relatorio = build_report(resultados, count_tables(spark, settings), settings)
caminho = save_report(relatorio)

print("Status:", relatorio["status"])
print("Resumo:", relatorio["resumo"])
print("Relatório salvo em:", caminho)

In [0]:
display(pd.DataFrame([
    {**r.to_dict(), "detalhes": json.dumps(r.detalhes, ensure_ascii=False, default=str)}
    for r in resultados
]))

In [0]:
# Este check deve falhar
teste = checks.check_range(
    spark.table("workspace.fc.gold_partidas_clima"), "gold_partidas_clima", "total_gols", 0, 3
)
print("Aprovado?", teste.aprovado)
print("Detalhes:", teste.detalhes)

In [0]:
raise_if_failed(relatorio)
print("Dados liberados para a camada final.")

In [0]:
print(caminho.read_text(encoding="utf-8")[:1500])